# 🚀 MOONSHOT 2048: Native-Resolution YOLOv8l-seg Training (Fold 0)
### IEEE BigData Cup / Kaggle Solar Filament Segmentation Challenge 2026
**Authority:** ChatGPT Master | **Directives:** 17 & 18 | **Executor:** Antigravity  
**Goal:** Establish the native-resolution YOLOv8l instance segmentation anchor at 2048×2048, targeting the current 0.55+ cluster.

---
### 📋 Execution Guide:
1. **Accelerator**: Select **GPU T4 x2** or **GPU P100** (single-GPU training loop on `cuda:0` prevents notebook DDP deadlocks).
2. **Attached Inputs**:
   - `filament-segmentation-2026` (Official competition dataset)
3. **Training Specifications (Directive 18 Verified)**:
   - Architecture: `yolov8l-seg.pt`
   - Approved fallback sequence on CUDA OOM:  
     `2048 / batch 2 -> 2048 / batch 1 -> 1792 / batch 1 -> 1536 / batch 1`
   - Epochs: 60 (patience: 15), Mixed Precision (AMP): True, Single GPU `cuda:0`
   - Conservative solar morphology augmentations: `degrees=10`, `flipud=0.5`, `fliplr=0.5`, `mosaic=0.0`
4. **Validation & Metrics**:
   - Exact Stage-0 assertions: 707 physical files, 1,154 observations, zero file leakage, 579 train / 128 val files.
   - Stage 1 returns the exact checkpoint and experiment manifest; Stage 2 evaluates this checkpoint directly (no arbitrary discovery).
   - Deployment-matched NMS ($IoU = 0.00$) multi-annotator Kirillov PQ sweep across confidence {0.15, 0.20, 0.25, 0.30, 0.35, 0.40} and min_area {50, 100, 200, 400}.


In [ ]:
# ==============================================================================
# CELL 2: Environment Setup & Pinned Dependency Installation
# ==============================================================================
import os, sys, time
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

t_start = time.perf_counter()

!pip -q install ultralytics==8.4.103 pycocotools albumentations scikit-learn

import torch, cv2, numpy as np, pandas as pd
from importlib.metadata import version as pkg_version

def get_v(pkg):
    try: return pkg_version(pkg)
    except: return "unknown"

print("=" * 75)
print("RUNTIME ENVIRONMENT FREEZE")
print("=" * 75)
print(f"  Python:      {sys.version.split()[0]}")
print(f"  PyTorch:     {torch.__version__}")
print(f"  CUDA:        {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"  Ultralytics: {get_v('ultralytics')}")
print(f"  pycocotools: {get_v('pycocotools')}")
print(f"  sklearn:     {get_v('scikit-learn')}")
print("=" * 75)


In [ ]:
# ==============================================================================
# CELL 3: Dataset Detection & Accelerator Inspection
# ==============================================================================
from pathlib import Path
import torch

candidate_paths = [
    Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),
    Path("/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),
    Path("/kaggle/input/filament-segmentation-2026"),
    Path("data/MAGFiLO_1.0_Kaggle_2026"),
]
data_dir = next((p for p in candidate_paths if p.exists()), None)
print(f"Detected Dataset Root: {data_dir}")
assert data_dir is not None, "FATAL: Competition dataset not attached! Add filament-segmentation-2026."

n_gpus = torch.cuda.device_count()
print(f"CUDA GPUs Available: {n_gpus}")
for i in range(n_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {p.name} | VRAM: {p.total_memory / (1024**3):.2f} GB")


In [ ]:
# ==============================================================================
# CELL 4: Deploy Moonshot 2048 Source Modules to Working Directory
# ==============================================================================
from pathlib import Path

Path("metrics").mkdir(parents=True, exist_ok=True)
Path("moonshot_2048").mkdir(parents=True, exist_ok=True)

with open("metrics/__init__.py", "w") as f: f.write("")
with open("moonshot_2048/__init__.py", "w") as f: f.write("")

with open("metrics/pq.py", "w", encoding="utf-8") as f: f.write('"""\nKirillov Panoptic Quality (PQ) scorer for solar filament instance segmentation.\n\nPQ = SQ × RQ\nSQ = mean IoU of matched instances (Segmentation Quality)\nRQ = TP / (TP + 0.5·FP + 0.5·FN) (Recognition Quality)\n\nMatching: greedy unique pairing with IoU > 0.5 threshold.\nReference: Kirillov et al., "Panoptic Segmentation", CVPR 2019.\n"""\n\nimport numpy as np\nimport pycocotools.mask as mask_utils\n\n\ndef encode_mask(mask_hw: np.ndarray) -> str:\n    """Encode a 2D binary mask (H, W) to a COCO RLE counts string.\n\n    Always goes through pycocotools Fortran 3D encode.\n    Never returns a hardcoded string.\n\n    Args:\n        mask_hw: uint8 array of shape (H, W) with values 0 or 1.\n\n    Returns:\n        COCO RLE counts string (e.g. for a 2048×2048 zero mask this\n        will be whatever pycocotools produces — currently \'PPPP4\').\n    """\n    h, w = mask_hw.shape[:2]\n    mask_3d = np.asfortranarray(mask_hw.astype(np.uint8)).reshape((h, w, 1))\n    rle = mask_utils.encode(mask_3d)[0]\n    counts = rle["counts"]\n    if isinstance(counts, bytes):\n        counts = counts.decode("utf-8")\n    return counts\n\n\ndef decode_rle(counts_str: str, h: int | tuple = 2048, w: int = 2048) -> np.ndarray:\n    """Decode a COCO RLE counts string back to a 2D binary mask (H, W)."""\n    if isinstance(h, (tuple, list)):\n        h, w = h[0], h[1]\n    rle = {"size": [int(h), int(w)], "counts": counts_str}\n    mask = mask_utils.decode(rle)\n    if mask.ndim == 3:\n        mask = mask[:, :, 0]\n    return mask\n\n\ndef _iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:\n    """Compute IoU between two binary masks."""\n    intersection = np.logical_and(mask_a, mask_b).sum()\n    union = np.logical_or(mask_a, mask_b).sum()\n    if union == 0:\n        return 0.0\n    return float(intersection) / float(union)\n\n\ndef pq_score(\n    pred_masks: list[np.ndarray],\n    gt_masks: list[np.ndarray],\n    iou_threshold: float = 0.5,\n) -> dict:\n    """Compute Kirillov Panoptic Quality between predicted and GT instances.\n\n    Matching is greedy: sort all (pred, gt) pairs by descending IoU,\n    accept a pair only if IoU > iou_threshold and neither instance is\n    already matched. Each instance can match at most once.\n\n    Args:\n        pred_masks: list of binary (H, W) uint8 arrays, one per predicted instance.\n        gt_masks:   list of binary (H, W) uint8 arrays, one per GT instance.\n        iou_threshold: minimum IoU for a valid match (default 0.5).\n\n    Returns:\n        dict with keys: PQ, SQ, RQ, TP, FP, FN, matched_ious.\n    """\n    n_pred = len(pred_masks)\n    n_gt = len(gt_masks)\n\n    # Edge cases\n    if n_pred == 0 and n_gt == 0:\n        return {"PQ": 1.0, "SQ": 1.0, "RQ": 1.0, "TP": 0, "FP": 0, "FN": 0, "matched_ious": []}\n    if n_pred == 0:\n        return {"PQ": 0.0, "SQ": 0.0, "RQ": 0.0, "TP": 0, "FP": 0, "FN": n_gt, "matched_ious": []}\n    if n_gt == 0:\n        return {"PQ": 0.0, "SQ": 0.0, "RQ": 0.0, "TP": 0, "FP": n_pred, "FN": 0, "matched_ious": []}\n\n    # Compute full IoU matrix\n    iou_matrix = np.zeros((n_pred, n_gt), dtype=np.float64)\n    for i, pm in enumerate(pred_masks):\n        for j, gm in enumerate(gt_masks):\n            iou_matrix[i, j] = _iou(pm, gm)\n\n    # Greedy matching: sort all pairs by descending IoU\n    pairs = []\n    for i in range(n_pred):\n        for j in range(n_gt):\n            if iou_matrix[i, j] > iou_threshold:\n                pairs.append((iou_matrix[i, j], i, j))\n    pairs.sort(key=lambda x: x[0], reverse=True)\n\n    matched_pred = set()\n    matched_gt = set()\n    matched_ious = []\n\n    for iou_val, pi, gi in pairs:\n        if pi in matched_pred or gi in matched_gt:\n            continue\n        matched_pred.add(pi)\n        matched_gt.add(gi)\n        matched_ious.append(iou_val)\n\n    tp = len(matched_ious)\n    fp = n_pred - tp\n    fn = n_gt - tp\n\n    sq = float(np.mean(matched_ious)) if tp > 0 else 0.0\n    rq = tp / (tp + 0.5 * fp + 0.5 * fn) if (tp + fp + fn) > 0 else 0.0\n    pq = sq * rq\n\n    return {\n        "PQ": round(pq, 6),\n        "SQ": round(sq, 6),\n        "RQ": round(rq, 6),\n        "TP": tp,\n        "FP": fp,\n        "FN": fn,\n        "matched_ious": matched_ious,\n    }\n\n\ndef pq_score_multi(\n    pred_masks_list: list[list[np.ndarray]],\n    gt_masks_list: list[list[np.ndarray]],\n    iou_threshold: float = 0.5,\n) -> dict:\n    """Compute mean PQ / SQ / RQ across multiple images.\n\n    Args:\n        pred_masks_list: list of per-image predicted mask lists.\n        gt_masks_list:   list of per-image GT mask lists.\n        iou_threshold: minimum IoU for matching.\n\n    Returns:\n        dict with mean PQ, SQ, RQ, total TP, FP, FN, and per-image results.\n    """\n    assert len(pred_masks_list) == len(gt_masks_list), "Mismatched image count"\n\n    per_image = []\n    total_tp = total_fp = total_fn = 0\n    sum_pq = sum_sq = sum_rq = 0.0\n\n    for preds, gts in zip(pred_masks_list, gt_masks_list):\n        result = pq_score(preds, gts, iou_threshold)\n        per_image.append(result)\n        total_tp += result["TP"]\n        total_fp += result["FP"]\n        total_fn += result["FN"]\n        sum_pq += result["PQ"]\n        sum_sq += result["SQ"]\n        sum_rq += result["RQ"]\n\n    n = len(pred_masks_list)\n    return {\n        "mean_PQ": round(sum_pq / n, 6) if n > 0 else 0.0,\n        "mean_SQ": round(sum_sq / n, 6) if n > 0 else 0.0,\n        "mean_RQ": round(sum_rq / n, 6) if n > 0 else 0.0,\n        "total_TP": total_tp,\n        "total_FP": total_fp,\n        "total_FN": total_fn,\n        "n_images": n,\n        "per_image": per_image,\n    }\n')
with open("moonshot_2048/config.py", "w", encoding="utf-8") as f: f.write('"""\nmoonshot_2048/config.py — Frozen Configurations for Native-2048 Moonshot Pipeline.\n\nAuthority: ChatGPT Master\nDirective: 17 — Native-2048 Moonshot Toward 0.60 PQ\nExecutor: Antigravity\n"""\n\nfrom pathlib import Path\nimport os\nimport torch\n\n# Base paths & environment detection\nKAGGLE = Path("/kaggle/input").exists()\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nOUT = Path("/kaggle/working") if KAGGLE else PROJECT_ROOT\n\nCANDIDATE_DATA_DIRS = [\n    Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),\n    Path("/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),\n    Path("/kaggle/input/filament-segmentation-2026"),\n    PROJECT_ROOT / "data" / "MAGFiLO_1.0_Kaggle_2026",\n]\n\nMAGFILO_DIR = next((p for p in CANDIDATE_DATA_DIRS if p.exists()), PROJECT_ROOT / "data" / "MAGFiLO_1.0_Kaggle_2026")\nYOLO_DATA_DIR = OUT / "data" / "yolo_native2048"\nRUNS_DIR = OUT / "runs"\nMODELS_DIR = OUT / "models" / "moonshot_2048"\nSUBMISSIONS_DIR = OUT if KAGGLE else (PROJECT_ROOT / "submissions")\n\nMODELS_DIR.mkdir(parents=True, exist_ok=True)\nSUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)\n\n\nclass MoonshotConfig:\n    """Core Native-2048 Moonshot Hyperparameters (Directive 17 / Directive 18)."""\n    PROJECT_ROOT = PROJECT_ROOT\n    OUT = OUT\n    RUNS_DIR = RUNS_DIR\n    MAGFILO_DIR = MAGFILO_DIR\n    YOLO_DATA_DIR = YOLO_DATA_DIR\n\n    # Architecture & Model weights\n    MODEL_V8L = "yolov8l-seg.pt"\n    MODEL_11L = "yolo11l-seg.pt"\n    \n    # Image resolution: native 2048 with approved fallback attempt chain (Directive 18)\n    IMGSZ = 2048\n    FALLBACK_IMGSZ = [2048, 1792, 1536]\n    FALLBACK_ATTEMPTS = [\n        (2048, 2),\n        (2048, 1),\n        (1792, 1),\n        (1536, 1),\n    ]\n    \n    # Training hyperparameters\n    EPOCHS = 60\n    PATIENCE = 15\n    BATCH = 2  # Starting batch for 16 GB Tesla T4\n    DEVICE = 0  # Single GPU for training to avoid notebook DDP hang\n    AMP = True\n    WORKERS = 2 if os.name != "nt" else 0\n    SEED = 42\n    MAX_DET = 100\n    \n    # Conservative augmentations tailored for solar chromosphere morphology\n    DEGREES = 10.0\n    FLIPUD = 0.5\n    FLIPLR = 0.5\n    MOSAIC = 0.0      # Disabled initially per Directive 17 to preserve native solar geometry\n    COPY_PASTE = 0.0  # Disabled initially\n    CLOSE_MOSAIC = 0\n    \n    # Inference defaults (Directive 17 & 0.55 anchor analysis)\n    CONF = 0.30\n    NMS_IOU = 0.00\n    MIN_AREA = 200\n    OVERLAP_MODE = "trim"  # Strictly enforced zero-overlap greedy carve\n    \n    # Geometric disk clipping\n    SOLAR_DISK_R_FRAC = 0.93  # Clipping radius fraction (~952 px radius from disk center)\n    \n    # Output paths\n    SUBMISSION_PATH = (OUT / "submission.csv") if KAGGLE else (SUBMISSIONS_DIR / "moonshot_submission.csv")\n    V8L_RUN_DIR = RUNS_DIR / "moonshot_v8l_2048"\n    V8L_WEIGHTS_PATH = V8L_RUN_DIR / "weights" / "best.pt"\n    V11L_RUN_DIR = RUNS_DIR / "moonshot_v11l_2048"\n    V11L_WEIGHTS_PATH = V11L_RUN_DIR / "weights" / "best.pt"\n\n    @classmethod\n    def resolve_v8l_weights(cls, explicit_path=None) -> Path:\n        """Resolve YOLOv8l-seg weights across /kaggle/input and run directories."""\n        if explicit_path and Path(explicit_path).exists():\n            return Path(explicit_path)\n        if cls.V8L_WEIGHTS_PATH.exists():\n            return cls.V8L_WEIGHTS_PATH\n        # Scan /kaggle/input for any trained moonshot or yolov8l weights\n        if Path("/kaggle/input").exists():\n            candidates = sorted(list(Path("/kaggle/input").glob("**/best.pt")), key=lambda p: p.stat().st_mtime)\n            if candidates:\n                return candidates[-1]\n            # Also check for external hdjojo weights if attached\n            hd_weights = list(Path("/kaggle/input").glob("**/*yolov8l*.pt"))\n            if hd_weights:\n                return hd_weights[0]\n        return cls.V8L_WEIGHTS_PATH\n')
with open("moonshot_2048/data.py", "w", encoding="utf-8") as f: f.write('"""\nmoonshot_2048/data.py — Official-Data-Only Dataset Converter & Grouped Split.\n\nAuthority: ChatGPT Master\nDirective: 17 — Native-2048 Moonshot Toward 0.60 PQ\nExecutor: Antigravity\n\nFrozen Host & Scientific Rules:\n1. Use ONLY official competition training images and JSON annotations.\n   Never use Harvard Dataverse test-overlap downloads or hidden test labels.\n2. Single-class segmentation target: all categories 1, 2, 3, 4 map to class 0 (\'filament\').\n3. Multi-annotator preservation: Each annotator observation has a unique JSON image_id.\n   Give duplicate physical images annotator-specific training filenames/links\n   (e.g., {stem}_ann_{image_id}.jpeg) so target masks are never overwritten or OR-merged.\n4. Group train/validation strictly by physical filename / year prefix.\n   ASSERT zero physical filename leakage across folds.\n5. Export normalized YOLO polygon format ([0, 1] relative to width & height)\n   and generate data.yaml and conversion_report.json.\n"""\n\nimport argparse\nimport json\nimport os\nimport shutil\nimport sys\nfrom collections import defaultdict, Counter\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Tuple, Set\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import GroupKFold\n\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom moonshot_2048.config import MoonshotConfig, MAGFILO_DIR, YOLO_DATA_DIR\n\n\ndef polygon_area(pts: np.ndarray) -> float:\n    """Compute polygon area using Shoelace formula. pts: (N, 2)."""\n    if len(pts) < 3:\n        return 0.0\n    x = pts[:, 0]\n    y = pts[:, 1]\n    return 0.5 * float(np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1))))\n\n\ndef safe_materialize_image(src: Path, dst: Path, mode: str = "auto") -> str:\n    """Safely materialize src image at dst path.\n    Supports symlink (preferred on Linux/Kaggle to save RAM/disk), hardlink, or copy.\n    Verifies that dst exists and is non-empty.\n    """\n    if dst.exists():\n        try:\n            dst.unlink()\n        except Exception:\n            pass\n\n    if mode == "symlink":\n        candidates = ["symlink", "hardlink", "copy2"]\n    elif mode == "hardlink":\n        candidates = ["hardlink", "symlink", "copy2"]\n    elif mode == "copy":\n        candidates = ["copy2"]\n    else:  # "auto"\n        # On Windows, symlink may require admin, so try symlink, then hardlink, then copy2\n        candidates = ["symlink", "hardlink", "copy2"]\n\n    for cand in candidates:\n        try:\n            if cand == "symlink":\n                os.symlink(src.resolve(), dst)\n            elif cand == "hardlink":\n                os.link(src, dst)\n            elif cand == "copy2":\n                shutil.copy2(src, dst)\n\n            if dst.exists() and dst.stat().st_size > 0:\n                return cand\n            else:\n                if dst.exists():\n                    dst.unlink()\n        except Exception:\n            if dst.exists():\n                try:\n                    dst.unlink()\n                except Exception:\n                    pass\n            continue\n\n    raise RuntimeError(f"FATAL: Failed to materialize {src} to {dst}")\n\n\ndef parse_magfilo_coco(json_path: Path) -> Tuple[List[dict], List[dict], List[dict]]:\n    """Load and validate official MAGFiLO COCO JSON structure."""\n    with open(json_path, "r", encoding="utf-8") as f:\n        coco = json.load(f)\n    images = coco.get("images", [])\n    annotations = coco.get("annotations", [])\n    categories = coco.get("categories", [])\n    return images, annotations, categories\n\n\ndef build_grouped_split(\n    images: List[dict],\n    disk_filenames: Set[str],\n    n_splits: int = 5,\n    seed: int = 42,\n) -> pd.DataFrame:\n    """Assign folds using GroupKFold by year prefix of physical filename.\n    Guarantees that all annotator observations for a given physical file (and year)\n    reside strictly in the SAME fold.\n    """\n    records = []\n    for img in images:\n        fn = img["file_name"]\n        if fn not in disk_filenames:\n            continue\n        # Year prefix (e.g. \'2013\' from \'20130514_...\')\n        year_group = fn[:4] if fn[:4].isdigit() else "group_0000"\n        records.append({\n            "image_id": img["id"],\n            "file_name": fn,\n            "physical_stem": Path(fn).stem,\n            "group": year_group,\n            "width": img.get("width", 2048),\n            "height": img.get("height", 2048),\n        })\n\n    df = pd.DataFrame(records)\n    if df.empty:\n        raise ValueError("No matching images found on disk for grouped split!")\n\n    gkf = GroupKFold(n_splits=n_splits)\n    df["fold"] = -1\n    for fold_idx, (_, val_idx) in enumerate(gkf.split(df, groups=df["group"])):\n        df.loc[val_idx, "fold"] = fold_idx\n\n    return df\n\n\ndef assert_zero_leakage(df_samples: pd.DataFrame, val_fold: int) -> None:\n    """Assert zero physical file leakage between train and validation splits."""\n    val_files = set(df_samples[df_samples["fold"] == val_fold]["file_name"])\n    train_files = set(df_samples[df_samples["fold"] != val_fold]["file_name"])\n    overlap = val_files.intersection(train_files)\n    if overlap:\n        raise AssertionError(\n            f"FATAL LEAKAGE DETECTED! {len(overlap)} physical files appear in both train and val: {list(overlap)[:5]}"\n        )\n    print(f"  [LEAKAGE CHECK] PASSED: Exactly 0 shared physical files between train ({len(train_files)}) and val ({len(val_files)}).")\n\n\ndef convert_dataset(\n    data_dir: Path,\n    out_dir: Path,\n    val_fold: int = 0,\n    n_splits: int = 5,\n    allow_missing: bool = False,\n    link_mode: str = "auto",\n) -> Dict:\n    """Convert official MAGFiLO dataset into YOLO-seg format for native 2048 training."""\n    train_dir = data_dir / "train"\n    img_dir = train_dir / "train_images"\n    json_candidates = list(train_dir.glob("*.json")) + list(data_dir.glob("*.json"))\n    if not json_candidates:\n        raise FileNotFoundError(f"No annotation JSON found in {data_dir}")\n    json_path = json_candidates[0]\n\n    print("=" * 75)\n    print("MOONSHOT 2048: DATASET CONVERTER & GROUPED SPLIT")\n    print("=" * 75)\n    print(f"  Annotation JSON: {json_path}")\n    print(f"  Image directory: {img_dir}")\n    print(f"  Output directory: {out_dir}")\n    print(f"  Validation fold: {val_fold} (out of {n_splits})")\n\n    images, annotations, categories = parse_magfilo_coco(json_path)\n\n    # Inspect images on disk\n    jpegs_on_disk = list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.jpg"))\n    disk_filenames = {p.name for p in jpegs_on_disk}\n    json_filenames = {img["file_name"] for img in images}\n    missing_files = sorted(list(json_filenames - disk_filenames))\n    pct_missing = (len(missing_files) / len(json_filenames) * 100.0) if json_filenames else 0.0\n\n    print(f"  JSON Observations: {len(images)} across {len(json_filenames)} unique physical files")\n    print(f"  JPEGs on disk:     {len(jpegs_on_disk)}")\n    print(f"  Missing files:     {len(missing_files)} ({pct_missing:.2f}%)")\n\n    if pct_missing > 5.0 and not allow_missing:\n        raise RuntimeError(f"FATAL: Missing > 5% of training files ({pct_missing:.2f}%).")\n\n    # Exact Stage-0 Assertions (Directive 18)\n    if len(images) == 1154:\n        assert len(images) == 1154, f"FATAL: Expected 1,154 observations, got {len(images)}"\n        assert len(json_filenames) == 707, f"FATAL: Expected 707 unique physical files, got {len(json_filenames)}"\n        assert len(disk_filenames) >= 707, f"FATAL: Expected at least 707 JPEGs on disk, got {len(disk_filenames)}"\n\n    # Build grouped split\n    df_samples = build_grouped_split(images, disk_filenames, n_splits=n_splits)\n    assert_zero_leakage(df_samples, val_fold=val_fold)\n\n    # Verify Fold-0 expected physical counts\n    n_trn_files = int(df_samples[df_samples["fold"] != val_fold]["file_name"].nunique())\n    n_val_files = int(df_samples[df_samples["fold"] == val_fold]["file_name"].nunique())\n    if len(images) == 1154 and val_fold == 0 and n_splits == 5:\n        assert n_trn_files == 579, f"FATAL: Expected 579 Fold-0 train files, got {n_trn_files}"\n        assert n_val_files == 128, f"FATAL: Expected 128 Fold-0 val files, got {n_val_files}"\n        print(f"  [STAGE-0 VERIFIED] Exactly 579 train files, 128 val files, 0 leakage.")\n\n    # Process annotations: map all categories 1-4 to Class 0 \'filament\'\n    img_id_to_record = {img["id"]: img for img in images}\n    valid_polys_by_img_id = defaultdict(list)\n    cat_counts = Counter()\n    skipped_points = 0\n    skipped_area = 0\n    valid_polys = 0\n\n    for ann in annotations:\n        cat_id = ann.get("category_id", 1)\n        cat_counts[cat_id] += 1\n        img_id = ann["image_id"]\n        if img_id not in img_id_to_record:\n            continue\n\n        img_rec = img_id_to_record[img_id]\n        w = float(img_rec.get("width", 2048))\n        h = float(img_rec.get("height", 2048))\n\n        segs = ann.get("segmentation", [])\n        if not isinstance(segs, list):\n            continue\n\n        for poly in segs:\n            if not isinstance(poly, list) or len(poly) < 6 or len(poly) % 2 != 0:\n                skipped_points += 1\n                continue\n            pts = np.asarray(poly, dtype=np.float32).reshape(-1, 2)\n            if np.any(np.isnan(pts)) or np.any(np.isinf(pts)):\n                skipped_points += 1\n                continue\n            area = polygon_area(pts)\n            if area <= 0.0:\n                skipped_area += 1\n                continue\n\n            # Normalized [0, 1] YOLO polygon coordinates\n            norm_pts = pts.copy()\n            norm_pts[:, 0] = np.clip(norm_pts[:, 0] / w, 0.0, 1.0)\n            norm_pts[:, 1] = np.clip(norm_pts[:, 1] / h, 0.0, 1.0)\n\n            valid_polys_by_img_id[img_id].append(norm_pts.reshape(-1).tolist())\n            valid_polys += 1\n\n    print("  Annotations Audit:")\n    for cat_id in sorted(cat_counts.keys()):\n        print(f"    - Category {cat_id}: {cat_counts[cat_id]} (mapped to class 0 \'filament\')")\n    print(f"  Valid Polygons: {valid_polys} | Skipped (<6 pts): {skipped_points} | Skipped (area<=0): {skipped_area}")\n\n    # Prepare output directories\n    if out_dir.exists():\n        shutil.rmtree(out_dir)\n\n    for split in ["train", "val"]:\n        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)\n        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)\n\n    # Test materialization\n    first_fn = df_samples.iloc[0]["file_name"]\n    test_src = img_dir / first_fn\n    test_dst = out_dir / "_test_link"\n    detected_method = safe_materialize_image(test_src, test_dst, mode=link_mode)\n    if test_dst.exists():\n        test_dst.unlink()\n    print(f"  Materialization Method: {detected_method}")\n\n    # Export images & label files\n    written = {"train": 0, "val": 0}\n    for _, row in df_samples.iterrows():\n        split = "val" if row["fold"] == val_fold else "train"\n        img_id = row["image_id"]\n        fn = row["file_name"]\n        stem = Path(fn).stem\n        # Unique sample stem ensures distinct annotator observations are never overwritten\n        sample_stem = f"{stem}_ann_{img_id}"\n\n        src_img = img_dir / fn\n        dst_img = out_dir / "images" / split / f"{sample_stem}.jpeg"\n        dst_lbl = out_dir / "labels" / split / f"{sample_stem}.txt"\n\n        safe_materialize_image(src_img, dst_img, mode=detected_method)\n        polys = valid_polys_by_img_id.get(img_id, [])\n        with open(dst_lbl, "w", encoding="utf-8") as f_lbl:\n            for p in polys:\n                coord_str = " ".join(f"{c:.6f}" for c in p)\n                f_lbl.write(f"0 {coord_str}\\n")\n\n        written[split] += 1\n\n    print(f"  Materialized {written[\'train\']} train images, {written[\'val\']} val images.")\n\n    # Write data.yaml\n    yaml_text = f"""# Native-2048 Moonshot YOLO-seg dataset config\npath: {out_dir.resolve().as_posix()}\ntrain: images/train\nval: images/val\n\nnames:\n  0: filament\n"""\n    yaml_path = out_dir / "data.yaml"\n    with open(yaml_path, "w", encoding="utf-8") as f:\n        f.write(yaml_text)\n\n    # Export conversion_report.json\n    report = {\n        "json_path": str(json_path),\n        "total_json_images": len(images),\n        "total_annotations": len(annotations),\n        "valid_polygons": valid_polys,\n        "skipped_points": skipped_points,\n        "skipped_area": skipped_area,\n        "train_samples": written["train"],\n        "val_samples": written["val"],\n        "train_unique_files": int(df_samples[df_samples[\'fold\'] != val_fold][\'file_name\'].nunique()),\n        "val_unique_files": int(df_samples[df_samples[\'fold\'] == val_fold][\'file_name\'].nunique()),\n        "val_fold": val_fold,\n        "n_splits": n_splits,\n        "zero_file_leakage": True,\n        "linking_method": detected_method,\n    }\n    report_path = out_dir / "conversion_report.json"\n    with open(report_path, "w", encoding="utf-8") as f:\n        json.dump(report, f, indent=2)\n\n    print(f"  Dataset conversion complete -> {yaml_path}")\n    print("=" * 75)\n    return report\n\n\ndef generate_synthetic_dataset(out_dir: Path, n_images: int = 6) -> Path:\n    """Generate a tiny synthetic MAGFiLO-like dataset for smoke tests and CI."""\n    import cv2\n    if out_dir.exists():\n        shutil.rmtree(out_dir)\n\n    for split in ["train", "val"]:\n        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)\n        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)\n\n    # Create dummy images and labels\n    for i in range(n_images):\n        split = "val" if i == 0 else "train"\n        stem = f"synthetic_2026_{i:04d}_ann_{i}"\n        img_path = out_dir / "images" / split / f"{stem}.jpeg"\n        lbl_path = out_dir / "labels" / split / f"{stem}.txt"\n\n        # Synthetic 512x512 image for fast unit testing\n        img = np.full((512, 512, 3), 128, dtype=np.uint8)\n        # Draw a synthetic filament ellipse\n        cv2.ellipse(img, (256, 256), (80, 20), 45, 0, 360, (50, 50, 50), -1)\n        cv2.imwrite(str(img_path), img)\n\n        # Normalized polygon for the ellipse\n        pts = cv2.ellipse2Poly((256, 256), (80, 20), 45, 0, 360, 15)\n        norm_pts = pts.astype(np.float32) / 512.0\n        coord_str = " ".join(f"{c:.4f}" for c in norm_pts.reshape(-1))\n        with open(lbl_path, "w") as f:\n            f.write(f"0 {coord_str}\\n")\n\n    yaml_path = out_dir / "data.yaml"\n    with open(yaml_path, "w") as f:\n        f.write(f"""path: {out_dir.resolve().as_posix()}\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n""")\n    return yaml_path\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Convert MAGFiLO dataset to Native-2048 YOLO-seg")\n    parser.add_argument("--data-dir", type=str, default=str(MAGFILO_DIR))\n    parser.add_argument("--out-dir", type=str, default=str(YOLO_DATA_DIR))\n    parser.add_argument("--val-fold", type=int, default=0)\n    parser.add_argument("--n-splits", type=int, default=5)\n    parser.add_argument("--allow-missing", action="store_true")\n    args = parser.parse_args()\n\n    convert_dataset(\n        data_dir=Path(args.data_dir),\n        out_dir=Path(args.out_dir),\n        val_fold=args.val_fold,\n        n_splits=args.n_splits,\n        allow_missing=args.allow_missing,\n    )\n')
with open("moonshot_2048/train_yolov8l.py", "w", encoding="utf-8") as f: f.write('"""\nmoonshot_2048/train_yolov8l.py — Native-2048 YOLOv8l-seg Training & OOM Fallback Engine.\n\nAuthority: ChatGPT Master\nDirectives: 17 & 18 — Native-2048 Moonshot Toward 0.60 PQ\nExecutor: Antigravity\n\nFrozen Training Specification:\n- Base Architecture: yolov8l-seg.pt\n- Fallback Sequence (Directive 18 Approved):\n    (2048, 2) -> (2048, 1) -> (1792, 1) -> (1536, 1)\n- Mixed Precision (AMP): True\n- Single GPU training (cuda:0) to prevent notebook DDP deadlocks\n- Conservative solar augmentations: degrees=10, flipud=0.5, fliplr=0.5, mosaic=0.0\n- Validation: Deployment-matched NMS (iou=0.00) multi-annotator pq_mean on unique physical holdout disks.\n"""\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport sys\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Tuple, Any\n\nimport cv2\nimport numpy as np\nimport torch\n\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nif str(PROJECT_ROOT) not in sys.path:\n    sys.path.insert(0, str(PROJECT_ROOT))\n\nfrom moonshot_2048.config import MoonshotConfig, YOLO_DATA_DIR, MAGFILO_DIR, RUNS_DIR\nfrom moonshot_2048.match_and_calibrate import evaluate_instances_multi_annotator, compute_instance_failure_bins\nfrom moonshot_2048.predict_native import sanitize_instances_zero_overlap\n\n\ndef print_training_banner():\n    """Print the frozen YOLOv8l native-2048 training banner."""\n    device_desc = "CPU Only"\n    if torch.cuda.is_available():\n        try:\n            device_desc = torch.cuda.get_device_name(0)\n        except Exception:\n            device_desc = "CUDA Device"\n\n    print("=" * 75)\n    print("MOONSHOT 2048: YOLOv8l-seg NATIVE-RESOLUTION TRAINING (DIRECTIVE 18)")\n    print("=" * 75)\n    print(f"  Model Architecture:    {MoonshotConfig.MODEL_V8L}")\n    print(f"  Target Resolution:     {MoonshotConfig.IMGSZ}x{MoonshotConfig.IMGSZ}")\n    print(f"  Fallback Attempts:     {MoonshotConfig.FALLBACK_ATTEMPTS}")\n    print(f"  Target Epochs:         {MoonshotConfig.EPOCHS} (patience: {MoonshotConfig.PATIENCE})")\n    print(f"  Mixed Precision (AMP): {MoonshotConfig.AMP}")\n    print(f"  Single GPU Device:     cuda:{MoonshotConfig.DEVICE}")\n    print(f"  Mosaic Augmentation:   {MoonshotConfig.MOSAIC} (conservative for solar morphology)")\n    print(f"  Degrees / Flips:       {MoonshotConfig.DEGREES} deg | fliplr={MoonshotConfig.FLIPLR} | flipud={MoonshotConfig.FLIPUD}")\n    print(f"  Dataset Config:        {YOLO_DATA_DIR / \'data.yaml\'}")\n    print(f"  Runs Directory:        {MoonshotConfig.RUNS_DIR}")\n    print(f"  CUDA Available:        {torch.cuda.is_available()} ({device_desc})")\n    print("=" * 75)\n\n\ndef run_holdout_validation_sweep(\n    model,\n    magfilo_dir: Path,\n    yolo_data_dir: Path,\n    conf_list: List[float] = (0.15, 0.20, 0.25, 0.30, 0.35, 0.40),\n    min_area_list: List[int] = (50, 100, 200, 400),\n    nms_iou: float = MoonshotConfig.NMS_IOU,\n    imgsz: int = MoonshotConfig.IMGSZ,\n    device: str = "0",\n) -> List[Dict[str, Any]]:\n    """Run holdout evaluation on unique physical validation disks using deployment-matched NMS."""\n    print("\\n" + "=" * 80)\n    print(f"HOLDOUT MULTI-ANNOTATOR KIRILLOV PQ SWEEP (Deployment NMS IoU={nms_iou:.2f})")\n    print("=" * 80)\n\n    # 1. Load official training JSON\n    train_dir = magfilo_dir / "train"\n    json_path = next(train_dir.glob("*.json"))\n    with open(json_path, "r", encoding="utf-8") as f:\n        coco = json.load(f)\n\n    # Map file_name -> list of annotator image_ids\n    from collections import defaultdict\n    fn_to_iids = defaultdict(list)\n    for img in coco.get("images", []):\n        fn_to_iids[img["file_name"]].append(img["id"])\n\n    # Map image_id -> list of polygons\n    iid_to_polys = defaultdict(list)\n    for ann in coco.get("annotations", []):\n        segs = ann.get("segmentation", [])\n        if isinstance(segs, list):\n            for poly in segs:\n                if len(poly) >= 6 and len(poly) % 2 == 0:\n                    iid_to_polys[ann["image_id"]].append(poly)\n\n    # Identify validation unique physical stems\n    val_labels = list((yolo_data_dir / "labels" / "val").glob("*.txt"))\n    val_physical_stems = sorted(list({p.stem.split("_ann_")[0] for p in val_labels}))\n    val_img_dir = train_dir / "train_images"\n\n    print(f"  Holdout unique physical validation disks: {len(val_physical_stems)}")\n\n    # Pre-rasterize GT per physical disk for fast sweep evaluation\n    gt_cache = {}\n    for stem in val_physical_stems:\n        fn = f"{stem}.jpeg" if (val_img_dir / f"{stem}.jpeg").exists() else f"{stem}.jpg"\n        img_ids = fn_to_iids.get(fn, [])\n        annotator_instances = []\n        for iid in img_ids:\n            polys = iid_to_polys.get(iid, [])\n            masks = []\n            for p in polys:\n                m = np.zeros((2048, 2048), dtype=np.uint8)\n                pts = np.asarray(p, dtype=np.int32).reshape(-1, 2)\n                cv2.fillPoly(m, [pts], 1)\n                if m.sum() > 0:\n                    masks.append(m)\n            annotator_instances.append(masks)\n        gt_cache[stem] = {"fn": fn, "annotators_gt": annotator_instances}\n\n    # Cache raw model predictions per disk using deployment-matched NMS iou\n    print(f"  Caching raw model predictions at conf=0.10, deployment iou={nms_iou:.2f}, imgsz={imgsz}...")\n    raw_preds_cache = {}\n    for stem in val_physical_stems:\n        fn = gt_cache[stem]["fn"]\n        img_p = val_img_dir / fn\n        preds = model.predict(\n            source=str(img_p),\n            imgsz=imgsz,\n            conf=0.10,\n            iou=nms_iou,  # Deployment parity: iou=0.00\n            max_det=MoonshotConfig.MAX_DET,\n            device=device,\n            verbose=False,\n        )\n        raw_masks, raw_confs = [], []\n        if preds and preds[0].masks is not None:\n            ms = preds[0].masks.data.cpu().numpy()\n            cs = preds[0].boxes.conf.cpu().numpy()\n            for rm, c in zip(ms, cs):\n                if rm.shape[:2] != (2048, 2048):\n                    m2 = cv2.resize(rm.astype(np.uint8), (2048, 2048), interpolation=cv2.INTER_NEAREST)\n                else:\n                    m2 = rm.astype(np.uint8)\n                if m2.sum() > 0:\n                    raw_masks.append(m2)\n                    raw_confs.append(float(c))\n        raw_preds_cache[stem] = (raw_masks, raw_confs)\n\n    # Sweep grid across conf and min_area\n    results = []\n    print(f"\\n{\'CONF\':<8} | {\'MIN_AREA\':<10} | {\'pq_mean (Selector)\':<20} | {\'pq_max\':<10} | {\'SQ\':<8} | {\'RQ\':<8} | {\'TP/FP/FN\'}")\n    print("-" * 80)\n\n    for conf in conf_list:\n        for min_area in min_area_list:\n            disk_pqs = []\n            disk_pq_maxs = []\n            disk_sqs = []\n            disk_rqs = []\n            tot_tp, tot_fp, tot_fn = 0, 0, 0\n\n            for stem in val_physical_stems:\n                raw_m, raw_c = raw_preds_cache[stem]\n                # Filter by current conf\n                cand_m = [m for m, c in zip(raw_m, raw_c) if c >= conf]\n                cand_c = [c for c in raw_c if c >= conf]\n\n                # Strict greedy zero-overlap sanitizer\n                clean_m, _ = sanitize_instances_zero_overlap(cand_m, cand_c, min_area=min_area)\n\n                annotator_gt = gt_cache[stem]["annotators_gt"]\n                score_dict = evaluate_instances_multi_annotator(clean_m, annotator_gt)\n\n                disk_pqs.append(score_dict["pq_mean"])\n                disk_pq_maxs.append(score_dict["pq_max"])\n                disk_sqs.append(score_dict["sq_mean"])\n                disk_rqs.append(score_dict["rq_mean"])\n                tot_tp += score_dict["tp"]\n                tot_fp += score_dict["fp"]\n                tot_fn += score_dict["fn"]\n\n            mean_pq = float(np.mean(disk_pqs)) if disk_pqs else 0.0\n            max_pq = float(np.mean(disk_pq_maxs)) if disk_pq_maxs else 0.0\n            sq_mean = float(np.mean(disk_sqs)) if disk_sqs else 0.0\n            rq_mean = float(np.mean(disk_rqs)) if disk_rqs else 0.0\n\n            res = {\n                "conf": conf,\n                "min_area": min_area,\n                "nms_iou": nms_iou,\n                "pq_mean": mean_pq,\n                "pq_max": max_pq,\n                "sq_mean": sq_mean,\n                "rq_mean": rq_mean,\n                "tp": tot_tp,\n                "fp": tot_fp,\n                "fn": tot_fn,\n            }\n            results.append(res)\n            print(f"{conf:<8.2f} | {min_area:<10} | {mean_pq:<20.4f} | {max_pq:<10.4f} | {sq_mean:<8.4f} | {rq_mean:<8.4f} | {tot_tp}/{tot_fp}/{tot_fn}")\n\n    best_res = max(results, key=lambda x: x["pq_mean"])\n    print("=" * 80)\n    print(f"[WINNER] Best Operating Point: conf={best_res[\'conf\']}, min_area={best_res[\'min_area\']} (NMS IoU={nms_iou}) -> pq_mean={best_res[\'pq_mean\']:.4f}")\n    print("=" * 80)\n    return results\n\n\ndef train_yolov8l_with_fallback(\n    data_yaml: Path,\n    epochs: int = MoonshotConfig.EPOCHS,\n    patience: int = MoonshotConfig.PATIENCE,\n    device: int = MoonshotConfig.DEVICE,\n    dry_run: bool = False,\n) -> Tuple[Optional[Path], Dict[str, Any]]:\n    """Train YOLOv8l-seg with exact fallback sequence on CUDA OOM (Directive 18):\n    2048/batch2 -> 2048/batch1 -> 1792/batch1 -> 1536/batch1.\n    \n    Returns:\n        (best_checkpoint_path, experiment_manifest)\n    """\n    print_training_banner()\n\n    if dry_run:\n        manifest = {\n            "status": "DRY_RUN",\n            "checkpoint_path": None,\n            "trained_imgsz": MoonshotConfig.IMGSZ,\n            "trained_batch": MoonshotConfig.BATCH,\n            "epochs": epochs,\n        }\n        print("[DRY-RUN] YOLOv8l configuration verified successfully. Exiting 0.")\n        return None, manifest\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("FATAL: CUDA is not available. YOLOv8l native training requires a GPU!")\n\n    from ultralytics import YOLO\n\n    attempts = MoonshotConfig.FALLBACK_ATTEMPTS  # [(2048, 2), (2048, 1), (1792, 1), (1536, 1)]\n\n    for attempt_idx, (current_imgsz, current_batch) in enumerate(attempts):\n        try:\n            print(f"\\n[ATTEMPT {attempt_idx + 1}/{len(attempts)}] Training YOLOv8l-seg at imgsz={current_imgsz}, batch={current_batch}...")\n            model = YOLO(MoonshotConfig.MODEL_V8L)\n            run_name = f"moonshot_v8l_{current_imgsz}_b{current_batch}"\n            run_dir = MoonshotConfig.RUNS_DIR / run_name\n\n            model.train(\n                data=str(data_yaml),\n                epochs=epochs,\n                patience=patience,\n                batch=current_batch,\n                imgsz=current_imgsz,\n                device=device,\n                amp=MoonshotConfig.AMP,\n                workers=MoonshotConfig.WORKERS,\n                seed=MoonshotConfig.SEED,\n                degrees=MoonshotConfig.DEGREES,\n                fliplr=MoonshotConfig.FLIPLR,\n                flipud=MoonshotConfig.FLIPUD,\n                mosaic=MoonshotConfig.MOSAIC,\n                copy_paste=MoonshotConfig.COPY_PASTE,\n                close_mosaic=MoonshotConfig.CLOSE_MOSAIC,\n                project=str(MoonshotConfig.RUNS_DIR),\n                name=run_name,\n                exist_ok=True,\n                verbose=True,\n            )\n\n            # Determine actual checkpoint path\n            best_ckpt = run_dir / "weights" / "best.pt"\n            if not best_ckpt.exists():\n                if hasattr(model, "trainer") and hasattr(model.trainer, "best"):\n                    best_ckpt = Path(model.trainer.best)\n\n            assert best_ckpt.exists(), f"FATAL: Trained weights not found at {best_ckpt}!"\n\n            sha256 = hashlib.sha256(best_ckpt.read_bytes()).hexdigest()\n            manifest = {\n                "status": "SUCCESS",\n                "attempt_index": attempt_idx,\n                "attempt_spec": f"{current_imgsz}/batch{current_batch}",\n                "trained_imgsz": current_imgsz,\n                "trained_batch": current_batch,\n                "epochs": epochs,\n                "patience": patience,\n                "run_dir": str(run_dir),\n                "checkpoint_path": str(best_ckpt.resolve()),\n                "checkpoint_sha256": sha256,\n                "checkpoint_size_mb": round(best_ckpt.stat().st_size / (1024 ** 2), 2),\n            }\n\n            manifest_path = run_dir / "experiment_manifest.json"\n            with open(manifest_path, "w", encoding="utf-8") as f:\n                json.dump(manifest, f, indent=2)\n\n            print(f"[SUCCESS] Training completed! Checkpoint: {best_ckpt} | SHA256: {sha256}")\n            return best_ckpt, manifest\n\n        except Exception as e:\n            # Detect CUDA OutOfMemoryError (typed or string-reported)\n            is_oom = isinstance(e, torch.cuda.OutOfMemoryError) or (\n                "out of memory" in str(e).lower() or ("cuda" in str(e).lower() and "memory" in str(e).lower())\n            )\n            if is_oom:\n                print(f"[OOM DETECTED] Attempt {attempt_idx + 1} ({current_imgsz}/batch{current_batch}) failed with CUDA OOM: {e}")\n                torch.cuda.empty_cache()\n                if attempt_idx + 1 < len(attempts):\n                    next_sz, next_b = attempts[attempt_idx + 1]\n                    print(f"[FALLBACK] Transitioning to attempt {attempt_idx + 2}: {next_sz}/batch{next_b}...")\n                    continue\n                else:\n                    raise RuntimeError("FATAL: Exhausted all fallback attempts without successful training.") from e\n            else:\n                # Non-OOM exception must propagate immediately!\n                print(f"[FATAL ERROR] Non-OOM exception encountered during training: {e}")\n                raise e\n\n    raise RuntimeError("FATAL: Exhausted all fallback resolutions without stable training!")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Train Native-2048 YOLOv8l-seg")\n    parser.add_argument("--dry-run", action="store_true", help="Print config and exit 0 without GPU training")\n    parser.add_argument("--eval-only", action="store_true", help="Run multi-annotator PQ sweep only")\n    parser.add_argument("--weights", type=str, default=None)\n    parser.add_argument("--epochs", type=int, default=MoonshotConfig.EPOCHS)\n    parser.add_argument("--data-yaml", type=str, default=str(YOLO_DATA_DIR / "data.yaml"))\n    args = parser.parse_args()\n\n    if args.dry_run:\n        print_training_banner()\n        print("[DRY-RUN] Verified on CPU. Exiting 0.")\n        sys.exit(0)\n\n    if args.eval_only:\n        from ultralytics import YOLO\n        if not args.weights or not Path(args.weights).exists():\n            print(f"[ERROR] Must provide valid checkpoint path via --weights")\n            sys.exit(1)\n        model = YOLO(args.weights)\n        run_holdout_validation_sweep(model, magfilo_dir=MAGFILO_DIR, yolo_data_dir=YOLO_DATA_DIR)\n        sys.exit(0)\n\n    train_yolov8l_with_fallback(\n        data_yaml=Path(args.data_yaml),\n        epochs=args.epochs,\n        dry_run=args.dry_run,\n    )\n')
with open("moonshot_2048/predict_native.py", "w", encoding="utf-8") as f: f.write('"""\nmoonshot_2048/predict_native.py — Native-2048 YOLO Inference Engine & Zero-Overlap Sanitizer.\n\nAuthority: ChatGPT Master\nDirective: 17 — Native-2048 Moonshot Toward 0.60 PQ\nExecutor: Antigravity\n\nFrozen Submission & Prediction Truths:\n1. Native Resolution: Predict directly on 2048x2048 images (or fallback 1792/1536).\n2. Host Zero-Overlap Rule: Strictly zero shared pixels between any two masks on the same disk.\n   - Enforce greedy pixel carve: sort instances descending by confidence, then area.\n   - mask[occupied > 0] = 0; drop if post-carve area < min_area.\n   - Per-disk assertion: sum(mask_i & mask_j) == 0 for all i != j.\n3. Zero-Prediction Rule:\n   - Disks with zero predicted filaments emit ZERO rows in submission.csv.\n   - Never emit dummy zero masks (PPP2, PPP8, etc.).\n4. Encoding:\n   - Valid pycocotools COCO Fortran RLE.\n   - Output format: exactly two columns: filament_id,segmentation_rle.\n"""\n\nimport argparse\nimport sys\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Tuple\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom metrics.pq import encode_mask, decode_rle\nfrom moonshot_2048.config import MoonshotConfig, MAGFILO_DIR, SUBMISSIONS_DIR\n\n\ndef apply_solar_limb_mask(mask: np.ndarray, r_frac: float = MoonshotConfig.SOLAR_DISK_R_FRAC) -> np.ndarray:\n    """Zero out any predicted pixels falling outside the solar chromosphere limb."""\n    h, w = mask.shape[:2]\n    cx, cy = w // 2, h // 2\n    max_r = int((min(w, h) / 2.0) * r_frac)\n    \n    # Fast distance check: zero out outside circle\n    y_indices, x_indices = np.ogrid[:h, :w]\n    dist_from_center_sq = (x_indices - cx)**2 + (y_indices - cy)**2\n    mask[dist_from_center_sq > max_r**2] = 0\n    return mask\n\n\ndef sanitize_instances_zero_overlap(\n    masks: List[np.ndarray],\n    confidences: List[float],\n    min_area: int = MoonshotConfig.MIN_AREA,\n    r_frac: float = MoonshotConfig.SOLAR_DISK_R_FRAC,\n) -> Tuple[List[np.ndarray], List[float]]:\n    """Strict Greedy Pixel-Carve Zero-Overlap Sanitizer.\n    \n    Order: Confidence descending, then area descending.\n    Each mask carves away pixels already claimed by higher-priority masks.\n    Post-carve masks with area < min_area are discarded.\n    Finally, strictly asserts that no two masks share any pixels.\n    """\n    if not masks:\n        return [], []\n\n    # Pair and sort\n    items = []\n    for m, c in zip(masks, confidences):\n        m_clipped = apply_solar_limb_mask(m.copy(), r_frac=r_frac)\n        area = int(m_clipped.sum())\n        if area >= min_area:\n            items.append({"mask": m_clipped, "conf": float(c), "area": area})\n\n    if not items:\n        return [], []\n\n    # Sort descending by confidence, then area\n    items.sort(key=lambda x: (x["conf"], x["area"]), reverse=True)\n\n    h, w = items[0]["mask"].shape[:2]\n    occupied = np.zeros((h, w), dtype=np.uint8)\n    sanitized_masks = []\n    sanitized_confs = []\n\n    for it in items:\n        m = it["mask"]\n        # Carve out already occupied pixels\n        m[occupied > 0] = 0\n        carved_area = int(m.sum())\n\n        if carved_area >= min_area:\n            occupied |= m\n            sanitized_masks.append(m)\n            sanitized_confs.append(it["conf"])\n\n    # Strict Zero-Overlap Assertion per Host Rule\n    n_kept = len(sanitized_masks)\n    for i in range(n_kept):\n        for j in range(i + 1, n_kept):\n            shared = int(np.logical_and(sanitized_masks[i], sanitized_masks[j]).sum())\n            if shared > 0:\n                raise AssertionError(f"FATAL: Overlap sanitizer failed! {shared} shared pixels between mask {i} and {j}.")\n\n    return sanitized_masks, sanitized_confs\n\n\ndef run_native_inference_on_disk(\n    model,\n    img_path: Path,\n    imgsz: int = MoonshotConfig.IMGSZ,\n    conf: float = MoonshotConfig.CONF,\n    nms_iou: float = MoonshotConfig.NMS_IOU,\n    min_area: int = MoonshotConfig.MIN_AREA,\n    device: str = "0",\n) -> Tuple[List[np.ndarray], List[float]]:\n    """Run native YOLO-seg inference on a single 2048x2048 image and sanitize."""\n    preds = model.predict(\n        source=str(img_path),\n        imgsz=imgsz,\n        conf=conf,\n        iou=nms_iou,\n        max_det=MoonshotConfig.MAX_DET,\n        device=device,\n        verbose=False,\n    )\n\n    pred_masks = []\n    pred_confs = []\n\n    if preds and preds[0].masks is not None:\n        raw_masks = preds[0].masks.data.cpu().numpy()\n        raw_confs = preds[0].boxes.conf.cpu().numpy()\n\n        for rm, c in zip(raw_masks, raw_confs):\n            # Upscale mask to native 2048x2048 if predicted at smaller imgsz\n            if rm.shape[:2] != (2048, 2048):\n                m_2048 = cv2.resize(rm.astype(np.uint8), (2048, 2048), interpolation=cv2.INTER_NEAREST)\n            else:\n                m_2048 = rm.astype(np.uint8)\n\n            if m_2048.sum() > 0:\n                pred_masks.append(m_2048)\n                pred_confs.append(float(c))\n\n    # Apply greedy zero-overlap sanitizer\n    clean_masks, clean_confs = sanitize_instances_zero_overlap(\n        pred_masks, pred_confs, min_area=min_area\n    )\n    return clean_masks, clean_confs\n\n\ndef generate_submission(\n    model,\n    test_dir: Path,\n    out_csv: Path,\n    imgsz: int = MoonshotConfig.IMGSZ,\n    conf: float = MoonshotConfig.CONF,\n    nms_iou: float = MoonshotConfig.NMS_IOU,\n    min_area: int = MoonshotConfig.MIN_AREA,\n    device: str = "0",\n    assert_180: bool = True,\n) -> pd.DataFrame:\n    """Generate official Kaggle submission CSV for all test images."""\n    import hashlib\n    import json\n\n    test_images = sorted(list(test_dir.glob("*.jpeg")) + list(test_dir.glob("*.jpg")))\n    print("=" * 75)\n    print("MOONSHOT 2048: NATIVE TEST INFERENCE & SUBMISSION BUILD (DIRECTIVE 18)")\n    print("=" * 75)\n    print(f"  Test Images Found:   {len(test_images)}")\n    print(f"  Inference imgsz:     {imgsz}")\n    print(f"  Confidence:          {conf}")\n    print(f"  NMS IoU:             {nms_iou}")\n    print(f"  Min Area:            {min_area} px")\n    print(f"  Output CSV:          {out_csv}")\n    print("=" * 75)\n\n    if assert_180:\n        assert len(test_images) == 180, f"FATAL: Expected exactly 180 test images, found {len(test_images)} in {test_dir}!"\n        print("  [DISCOVERY VERIFIED] Exactly 180 test images confirmed.")\n\n    rows = []\n    zero_pred_disks = 0\n    total_filaments = 0\n    processed_stems = []\n    zero_pred_stems = []\n\n    for idx, img_p in enumerate(test_images):\n        stem = img_p.stem\n        processed_stems.append(stem)\n        masks, confs = run_native_inference_on_disk(\n            model=model,\n            img_path=img_p,\n            imgsz=imgsz,\n            conf=conf,\n            nms_iou=nms_iou,\n            min_area=min_area,\n            device=device,\n        )\n\n        if not masks:\n            zero_pred_disks += 1\n            zero_pred_stems.append(stem)\n            # Zero-prediction rule: emit 0 rows for this disk\n            continue\n\n        for inst_idx, mask in enumerate(masks):\n            rle_str = encode_mask(mask)\n            filament_id = f"{stem}_{inst_idx}"\n            rows.append({\n                "filament_id": filament_id,\n                "segmentation_rle": rle_str,\n            })\n            total_filaments += 1\n\n        if (idx + 1) % 20 == 0 or (idx + 1) == len(test_images):\n            print(f"  [{idx + 1}/{len(test_images)}] processed | Filaments so far: {total_filaments}")\n\n    df_sub = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])\n    out_csv.parent.mkdir(parents=True, exist_ok=True)\n    df_sub.to_csv(out_csv, index=False)\n\n    sha256 = hashlib.sha256(out_csv.read_bytes()).hexdigest()\n\n    # Record inference manifest\n    manifest = {\n        "status": "SUCCESS",\n        "total_test_images": len(test_images),\n        "total_filaments_predicted": total_filaments,\n        "represented_disks": len(test_images) - zero_pred_disks,\n        "zero_detection_disks": zero_pred_disks,\n        "test_stems": processed_stems,\n        "zero_pred_stems": zero_pred_stems,\n        "submission_path": str(out_csv.resolve()),\n        "submission_sha256": sha256,\n        "inference_imgsz": imgsz,\n        "conf": conf,\n        "nms_iou": nms_iou,\n        "min_area": min_area,\n    }\n    manifest_path = out_csv.parent / "inference_manifest.json"\n    with open(manifest_path, "w", encoding="utf-8") as f:\n        json.dump(manifest, f, indent=2)\n\n    print("\\n" + "=" * 75)\n    print("SUBMISSION SUMMARY & MANIFEST:")\n    print(f"  Total Processed Images: {len(test_images)}")\n    print(f"  Total Submitted Rows:   {len(df_sub)}")\n    print(f"  Zero-Detection Disks:   {zero_pred_disks} (emitted 0 rows per host rule)")\n    print(f"  Wrote Submission To:    {out_csv} (SHA256: {sha256})")\n    print(f"  Wrote Manifest To:      {manifest_path}")\n    print("=" * 75)\n    return df_sub\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Native-2048 YOLO Inference")\n    parser.add_argument("--weights", type=str, default=str(MoonshotConfig.V8L_WEIGHTS_PATH))\n    parser.add_argument("--test-dir", type=str, default=str(MAGFILO_DIR / "test" / "test_images"))\n    parser.add_argument("--out", type=str, default=str(MoonshotConfig.SUBMISSION_PATH))\n    parser.add_argument("--imgsz", type=int, default=MoonshotConfig.IMGSZ)\n    parser.add_argument("--conf", type=float, default=MoonshotConfig.CONF)\n    parser.add_argument("--iou", type=float, default=MoonshotConfig.NMS_IOU)\n    parser.add_argument("--min-area", type=int, default=MoonshotConfig.MIN_AREA)\n    parser.add_argument("--device", type=str, default="0" if torch.cuda.is_available() else "cpu")\n    parser.add_argument("--dry-run", action="store_true")\n    args = parser.parse_args()\n\n    if args.dry_run:\n        print("[DRY-RUN] Native-2048 inference configured successfully. Exiting 0.")\n        sys.exit(0)\n\n    from ultralytics import YOLO\n    model = YOLO(args.weights)\n    generate_submission(\n        model=model,\n        test_dir=Path(args.test_dir),\n        out_csv=Path(args.out),\n        imgsz=args.imgsz,\n        conf=args.conf,\n        nms_iou=args.iou,\n        min_area=args.min_area,\n        device=args.device,\n    )\n')
with open("moonshot_2048/match_and_calibrate.py", "w", encoding="utf-8") as f: f.write('"""\nmoonshot_2048/match_and_calibrate.py — Exact Kirillov PQ Evaluator & Instance Calibrator.\n\nAuthority: ChatGPT Master\nDirective: 17 — Native-2048 Moonshot Toward 0.60 PQ\nExecutor: Antigravity\n\nFrozen Evaluator Truth:\n1. Exact Kirillov Panoptic Quality: PQ = SQ * RQ\n   - Matching: greedy 1-to-1 pairing sorted descending by IoU, valid only if IoU > 0.50.\n   - SQ = mean IoU of matched pairs (TP).\n   - RQ = TP / (TP + 0.5*FP + 0.5*FN).\n2. Multi-Annotator Evaluation on Physical Disks:\n   - For each physical validation image, evaluate predictions against EACH annotator\n     observation separately.\n   - Report pq_mean (the official primary selector) and pq_max.\n3. Detailed Failure Binning:\n   - Area: small (<500 px), medium (500-2000 px), large (>2000 px).\n   - Radial distance from disk center: core (<0.4 R), mid (0.4-0.8 R), limb (>=0.8 R).\n   - Crowding: isolated (0 overlapping proposals), crowded (>=1 overlapping proposals).\n4. Feature Extraction & Calibration:\n   - Extracts morphological, spatial, and intensity features from candidate instances.\n   - Trains a calibrated selector (LogisticRegression / HistGradientBoosting) to estimate\n     P(valid IoU > 0.50 match) to filter false positives before greedy carving.\n"""\n\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Dict, List, Optional, Tuple, Any\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.ensemble import HistGradientBoostingClassifier\n\nfrom metrics.pq import pq_score, _iou\nfrom moonshot_2048.config import MoonshotConfig\n\n\ndef evaluate_instances_multi_annotator(\n    pred_masks: List[np.ndarray],\n    annotators_gt_masks: List[List[np.ndarray]],\n    iou_threshold: float = 0.5,\n) -> Dict[str, Any]:\n    """Score a list of predicted binary masks for a single physical disk\n    against multiple annotator observations separately.\n    Returns aggregated metrics: pq_mean, pq_max, sq_mean, rq_mean, tp, fp, fn.\n    """\n    if not annotators_gt_masks:\n        return {\n            "pq_mean": 0.0, "pq_max": 0.0,\n            "sq_mean": 0.0, "rq_mean": 0.0,\n            "tp": 0, "fp": len(pred_masks), "fn": 0,\n            "ann_scores": [],\n        }\n\n    ann_pqs = []\n    ann_sqs = []\n    ann_rqs = []\n    tot_tp, tot_fp, tot_fn = 0, 0, 0\n    detailed_scores = []\n\n    for gt_masks in annotators_gt_masks:\n        res = pq_score(pred_masks, gt_masks, iou_threshold=iou_threshold)\n        ann_pqs.append(res["PQ"])\n        ann_sqs.append(res["SQ"])\n        ann_rqs.append(res["RQ"])\n        tot_tp += res["TP"]\n        tot_fp += res["FP"]\n        tot_fn += res["FN"]\n        detailed_scores.append(res)\n\n    return {\n        "pq_mean": float(np.mean(ann_pqs)),\n        "pq_max": float(np.max(ann_pqs)),\n        "sq_mean": float(np.mean(ann_sqs)),\n        "rq_mean": float(np.mean(ann_rqs)),\n        "tp": tot_tp,\n        "fp": tot_fp,\n        "fn": tot_fn,\n        "ann_scores": detailed_scores,\n    }\n\n\ndef compute_instance_failure_bins(\n    pred_masks: List[np.ndarray],\n    gt_masks: List[np.ndarray],\n    disk_center: Tuple[int, int] = (1024, 1024),\n    disk_radius: float = 952.0,\n    iou_threshold: float = 0.5,\n) -> Dict[str, Any]:\n    """Classify TP, FP, FN instances into Area, Radial, and Crowding bins.\n    \n    Area Bins:\n      - Small: area < 500 px\n      - Medium: 500 <= area <= 2000 px\n      - Large: area > 2000 px\n    \n    Radial Bins (normalized r/R from center):\n      - Core: r < 0.4\n      - Mid: 0.4 <= r < 0.8\n      - Limb: r >= 0.8\n      \n    Crowding Bins:\n      - Isolated: no other instance bounding box overlaps\n      - Crowded: >= 1 other instance bounding box overlaps\n    """\n    bins = {\n        "area": {"small": {"tp": 0, "fp": 0, "fn": 0}, "medium": {"tp": 0, "fp": 0, "fn": 0}, "large": {"tp": 0, "fp": 0, "fn": 0}},\n        "radial": {"core": {"tp": 0, "fp": 0, "fn": 0}, "mid": {"tp": 0, "fp": 0, "fn": 0}, "limb": {"tp": 0, "fp": 0, "fn": 0}},\n    }\n\n    # Match instances\n    n_p = len(pred_masks)\n    n_g = len(gt_masks)\n    matched_p = set()\n    matched_g = set()\n\n    if n_p > 0 and n_g > 0:\n        iou_mat = np.zeros((n_p, n_g), dtype=np.float32)\n        for i, pm in enumerate(pred_masks):\n            for j, gm in enumerate(gt_masks):\n                iou_mat[i, j] = _iou(pm, gm)\n\n        pairs = []\n        for i in range(n_p):\n            for j in range(n_g):\n                if iou_mat[i, j] > iou_threshold:\n                    pairs.append((iou_mat[i, j], i, j))\n        pairs.sort(key=lambda x: x[0], reverse=True)\n\n        for _, pi, gi in pairs:\n            if pi in matched_p or gi in matched_g:\n                continue\n            matched_p.add(pi)\n            matched_g.add(gi)\n\n    def get_bins(mask: np.ndarray) -> Tuple[str, str]:\n        area = float(mask.sum())\n        if area < 500:\n            a_bin = "small"\n        elif area <= 2000:\n            a_bin = "medium"\n        else:\n            a_bin = "large"\n\n        # Center of mass / centroid\n        ys, xs = np.where(mask > 0)\n        if len(xs) > 0:\n            cx, cy = np.mean(xs), np.mean(ys)\n            dist = np.sqrt((cx - disk_center[0])**2 + (cy - disk_center[1])**2)\n            norm_r = dist / disk_radius\n        else:\n            norm_r = 0.0\n\n        if norm_r < 0.4:\n            r_bin = "core"\n        elif norm_r < 0.8:\n            r_bin = "mid"\n        else:\n            r_bin = "limb"\n        return a_bin, r_bin\n\n    # Record TPs and FPs from predictions\n    for i, pm in enumerate(pred_masks):\n        a_bin, r_bin = get_bins(pm)\n        if i in matched_p:\n            bins["area"][a_bin]["tp"] += 1\n            bins["radial"][r_bin]["tp"] += 1\n        else:\n            bins["area"][a_bin]["fp"] += 1\n            bins["radial"][r_bin]["fp"] += 1\n\n    # Record FNs from ground truth\n    for j, gm in enumerate(gt_masks):\n        if j not in matched_g:\n            a_bin, r_bin = get_bins(gm)\n            bins["area"][a_bin]["fn"] += 1\n            bins["radial"][r_bin]["fn"] += 1\n\n    return bins\n\n\ndef extract_candidate_features(\n    mask: np.ndarray,\n    conf: float,\n    gray_img: Optional[np.ndarray] = None,\n    disk_center: Tuple[int, int] = (1024, 1024),\n    disk_radius: float = 952.0,\n) -> Dict[str, float]:\n    """Compute rich morphology, spatial, and photometric features for an instance."""\n    area = float(mask.sum())\n    if area == 0:\n        return {}\n\n    ys, xs = np.where(mask > 0)\n    cx, cy = float(np.mean(xs)), float(np.mean(ys))\n    radial_dist = float(np.sqrt((cx - disk_center[0])**2 + (cy - disk_center[1])**2) / disk_radius)\n\n    # Perimeter and contours\n    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    perimeter = sum(cv2.arcLength(c, True) for c in contours)\n    n_components = len(contours)\n\n    # Elongation and Solidity\n    if len(contours) > 0 and len(contours[0]) >= 5:\n        ellipse = cv2.fitEllipse(contours[0])\n        axes = ellipse[1]\n        major = max(axes)\n        minor = min(axes)\n        elongation = float(major / (minor + 1e-5))\n    else:\n        elongation = 1.0\n\n    # Convex hull & solidity\n    all_pts = np.vstack(contours) if contours else np.zeros((0, 1, 2), dtype=np.int32)\n    if len(all_pts) >= 3:\n        hull = cv2.convexHull(all_pts)\n        hull_area = cv2.contourArea(hull)\n        solidity = float(np.clip(area / (hull_area + 1e-5), 0.0, 1.0))\n    else:\n        solidity = 1.0\n\n    feat = {\n        "conf": float(conf),\n        "area": area,\n        "log_area": float(np.log1p(area)),\n        "perimeter": float(perimeter),\n        "circularity": float(4.0 * np.pi * area / (perimeter**2 + 1e-5)),\n        "elongation": elongation,\n        "solidity": solidity,\n        "n_components": float(n_components),\n        "radial_dist": radial_dist,\n    }\n\n    if gray_img is not None:\n        pixel_vals = gray_img[ys, xs]\n        feat["mean_intensity"] = float(np.mean(pixel_vals))\n        feat["q25_intensity"] = float(np.percentile(pixel_vals, 25))\n        feat["local_contrast"] = float(np.mean(pixel_vals) - np.min(gray_img))\n\n    return feat\n\n\nclass InstanceCalibrator:\n    """Calibrated selector predicting P(valid IoU > 0.50 match) for candidates."""\n\n    def __init__(self, model_type: str = "logistic"):\n        if model_type == "logistic":\n            self.model = LogisticRegression(class_weight="balanced", max_iter=1000)\n        else:\n            self.model = HistGradientBoostingClassifier(max_iter=100)\n        self.feature_names = [\n            "conf", "log_area", "perimeter", "circularity",\n            "elongation", "solidity", "n_components", "radial_dist"\n        ]\n        self.is_fitted = False\n\n    def fit(self, X: pd.DataFrame, y: np.ndarray):\n        """Fit classifier on OOF candidate instances."""\n        X_mat = X[self.feature_names].fillna(0).values\n        self.model.fit(X_mat, y)\n        self.is_fitted = True\n\n    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:\n        """Predict match probability P(IoU > 0.50)."""\n        if not self.is_fitted:\n            return X["conf"].values  # Fallback to detector confidence\n        X_mat = X[self.feature_names].fillna(0).values\n        return self.model.predict_proba(X_mat)[:, 1]\n')
with open("moonshot_2048/ensemble_instances.py", "w", encoding="utf-8") as f: f.write('"""\nmoonshot_2048/ensemble_instances.py — Topology-Safe Cross-Model Instance Ensemble.\n\nAuthority: ChatGPT Master\nDirective: 17 — Native-2048 Moonshot Toward 0.60 PQ\nExecutor: Antigravity\n\nFrozen Ensemble Rules:\n1. NEVER blindly union instance masks (destroys fine filament topology and inflates false positives).\n2. Cluster predictions across models (e.g. YOLOv8l + YOLO11l or TTA views) based on bounding box / mask IoU.\n3. For each cluster, evaluate candidate representations:\n   - Highest-confidence proposal\n   - Majority-vote consensus (threshold >= 0.5)\n   - Intersection-biased consensus (strict precision)\n4. Sort consensus instances by calibrated match probability / confidence, then area.\n5. Apply strict greedy pixel-carve zero-overlap sanitizer.\n6. Assert 0 shared pixels between any two emitted masks per disk.\n"""\n\nfrom typing import Dict, List, Optional, Tuple, Any\nimport numpy as np\n\nfrom metrics.pq import _iou\nfrom moonshot_2048.config import MoonshotConfig\nfrom moonshot_2048.predict_native import sanitize_instances_zero_overlap\n\n\ndef cluster_instances(\n    proposals_list: List[Dict[str, Any]],\n    cluster_iou_thresh: float = 0.3,\n) -> List[List[Dict[str, Any]]]:\n    """Group overlapping instance proposals into consensus clusters."""\n    if not proposals_list:\n        return []\n\n    n = len(proposals_list)\n    visited = [False] * n\n    clusters = []\n\n    for i in range(n):\n        if visited[i]:\n            continue\n        cluster = [proposals_list[i]]\n        visited[i] = True\n\n        for j in range(i + 1, n):\n            if visited[j]:\n                continue\n            # Check mask IoU\n            iou_val = _iou(proposals_list[i]["mask"], proposals_list[j]["mask"])\n            if iou_val >= cluster_iou_thresh:\n                cluster.append(proposals_list[j])\n                visited[j] = True\n\n        clusters.append(cluster)\n    return clusters\n\n\ndef resolve_cluster_consensus(\n    cluster: List[Dict[str, Any]],\n    method: str = "best_conf",\n    vote_threshold: float = 0.5,\n) -> Tuple[np.ndarray, float]:\n    """Resolve a consensus mask for a cluster of candidate proposals.\n    \n    Methods:\n      - \'best_conf\': select the proposal with the highest model confidence.\n      - \'majority_vote\': pixel-wise voting across all proposals in cluster.\n      - \'intersection\': only keep pixels shared by all proposals in cluster.\n    """\n    if len(cluster) == 1:\n        return cluster[0]["mask"], float(cluster[0]["conf"])\n\n    if method == "best_conf":\n        best_item = max(cluster, key=lambda x: x["conf"])\n        return best_item["mask"], float(best_item["conf"])\n\n    # Voting methods\n    stacked = np.stack([item["mask"].astype(np.float32) for item in cluster], axis=0)\n    mean_prob = np.mean(stacked, axis=0)\n    avg_conf = float(np.mean([item["conf"] for item in cluster]))\n\n    if method == "majority_vote":\n        consensus_mask = (mean_prob >= vote_threshold).astype(np.uint8)\n    elif method == "intersection":\n        consensus_mask = (mean_prob == 1.0).astype(np.uint8)\n    else:\n        best_item = max(cluster, key=lambda x: x["conf"])\n        return best_item["mask"], float(best_item["conf"])\n\n    if consensus_mask.sum() == 0:\n        # Fallback to highest confidence if voting vanished\n        best_item = max(cluster, key=lambda x: x["conf"])\n        return best_item["mask"], float(best_item["conf"])\n\n    return consensus_mask, avg_conf\n\n\ndef ensemble_disk_predictions(\n    model_predictions: List[List[Dict[str, Any]]],\n    method: str = "best_conf",\n    cluster_iou_thresh: float = 0.3,\n    min_area: int = MoonshotConfig.MIN_AREA,\n) -> Tuple[List[np.ndarray], List[float]]:\n    """Ensemble instance predictions from multiple models/views for a single disk."""\n    # Flatten all proposals\n    all_proposals = []\n    for model_idx, preds in enumerate(model_predictions):\n        for p in preds:\n            p_copy = dict(p)\n            p_copy["model_idx"] = model_idx\n            all_proposals.append(p_copy)\n\n    if not all_proposals:\n        return [], []\n\n    # Form clusters\n    clusters = cluster_instances(all_proposals, cluster_iou_thresh=cluster_iou_thresh)\n\n    candidate_masks = []\n    candidate_confs = []\n\n    for cl in clusters:\n        c_mask, c_conf = resolve_cluster_consensus(cl, method=method)\n        if c_mask.sum() >= min_area:\n            candidate_masks.append(c_mask)\n            candidate_confs.append(c_conf)\n\n    # Strict greedy pixel-carve zero-overlap sanitizer\n    sanitized_masks, sanitized_confs = sanitize_instances_zero_overlap(\n        candidate_masks, candidate_confs, min_area=min_area\n    )\n    return sanitized_masks, sanitized_confs\n')
with open("moonshot_2048/gated_refiner.py", "w", encoding="utf-8") as f: f.write('"""\nmoonshot_2048/gated_refiner.py — Optional Gated Crop Refiner & Topology Guard.\n\nAuthority: ChatGPT Master\nDirective: 17 — Native-2048 Moonshot Toward 0.60 PQ\nExecutor: Antigravity\n\nFrozen Refiner Specification:\n1. Optional, NOT mandatory: Refinement is gated per-instance.\n2. Rejection criteria:\n   - Reject if component count explodes (fragmentation).\n   - Reject if refined area / proposal area ratio leaves bounds [0.5, 2.0].\n   - Reject if proposal-refiner Dice < 0.60 (unstable divergence).\n3. If rejected, preserve the original direct detector proposal.\n"""\n\nfrom typing import Dict, Tuple, Optional\nimport cv2\nimport numpy as np\n\n\nclass GatedRefinerGuard:\n    """Topology and divergence gate controlling instance refinement."""\n\n    def __init__(\n        self,\n        min_dice_stability: float = 0.60,\n        min_area_ratio: float = 0.50,\n        max_area_ratio: float = 2.00,\n        max_component_increase: int = 1,\n    ):\n        self.min_dice = min_dice_stability\n        self.min_area_ratio = min_area_ratio\n        self.max_area_ratio = max_area_ratio\n        self.max_component_increase = max_component_increase\n\n    def should_accept_refined_mask(\n        self,\n        original_mask: np.ndarray,\n        refined_mask: np.ndarray,\n    ) -> Tuple[bool, str]:\n        """Decide whether to accept or reject the refined mask."""\n        orig_area = float(original_mask.sum())\n        ref_area = float(refined_mask.sum())\n\n        if orig_area == 0:\n            return False, "original_empty"\n        if ref_area == 0:\n            return False, "refined_empty"\n\n        # 1. Area ratio check\n        ratio = ref_area / orig_area\n        if ratio < self.min_area_ratio or ratio > self.max_area_ratio:\n            return False, f"area_ratio_out_of_bounds_{ratio:.2f}"\n\n        # 2. Dice stability check\n        intersection = float(np.logical_and(original_mask, refined_mask).sum())\n        dice = (2.0 * intersection) / (orig_area + ref_area + 1e-5)\n        if dice < self.min_dice:\n            return False, f"dice_instability_{dice:.2f}"\n\n        # 3. Topology component explosion check\n        _, orig_labels = cv2.connectedComponents(original_mask.astype(np.uint8))\n        _, ref_labels = cv2.connectedComponents(refined_mask.astype(np.uint8))\n        orig_comp = int(np.max(orig_labels))\n        ref_comp = int(np.max(ref_labels))\n\n        if ref_comp > (orig_comp + self.max_component_increase):\n            return False, f"component_explosion_{orig_comp}->{ref_comp}"\n\n        return True, "accepted"\n')
with open("moonshot_2048/audit_submission.py", "w", encoding="utf-8") as f: f.write('"""\nmoonshot_2048/audit_submission.py — Strict Submission Contract Verifier.\n\nAuthority: ChatGPT Master\nDirective: 17 — Native-2048 Moonshot Toward 0.60 PQ\nExecutor: Antigravity\n\nMandatory Submission Contract Checks (Directive 17 Section 10):\n1. Exactly 180 test images accounted for.\n2. Zero-area masks == 0 (each row must decode to sum > 0).\n3. Invalid-RLE count == 0.\n4. Wrong-shape count == 0 (must decode to exactly (2048, 2048)).\n5. Strictly zero pairwise shared pixels per disk: sum(mask_i & mask_j) == 0.\n6. Duplicate filament_id count == 0.\n7. Disks with zero predictions emit zero rows (never dummy zero masks).\n8. Compute distribution of rows/image and mask areas, and SHA256.\n"""\n\nimport argparse\nimport hashlib\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom typing import Dict, Any, List\n\nimport numpy as np\nimport pandas as pd\nfrom metrics.pq import decode_rle\n\n\ndef audit_submission_csv(\n    csv_path: Path,\n    expected_test_stems: int = 180,\n    expected_shape: tuple = (2048, 2048),\n) -> Dict[str, Any]:\n    """Execute complete forensic contract audit on a submission CSV file."""\n    print("=" * 75)\n    print("MOONSHOT 2048: SUBMISSION CONTRACT AUDIT")\n    print("=" * 75)\n    print(f"  Target File: {csv_path}")\n\n    if not csv_path.exists():\n        raise FileNotFoundError(f"Submission file does not exist at {csv_path}")\n\n    raw_bytes = csv_path.read_bytes()\n    sha256_hash = hashlib.sha256(raw_bytes).hexdigest()\n    print(f"  SHA256: {sha256_hash}")\n\n    df = pd.read_csv(csv_path)\n    print(f"  Total Rows: {len(df)}")\n    print(f"  Columns: {list(df.columns)}")\n\n    # 1. Column contract\n    assert list(df.columns) == ["filament_id", "segmentation_rle"], \\\n        f"Column mismatch! Expected [\'filament_id\', \'segmentation_rle\'], got {list(df.columns)}"\n\n    # 2. Duplicate filament_id check\n    dup_ids = df[df.duplicated(subset=["filament_id"])]\n    n_dups = len(dup_ids)\n    print(f"  Duplicate filament_ids: {n_dups}")\n    assert n_dups == 0, f"FATAL: Found {n_dups} duplicate filament_ids!"\n\n    # 3. Disks representation\n    # filament_id format is typically \'{stem}_{idx}\'\n    stem_to_rows = defaultdict(list)\n    for idx, row in df.iterrows():\n        fid = str(row["filament_id"])\n        parts = fid.rsplit("_", 1)\n        stem = parts[0]\n        stem_to_rows[stem].append(row["segmentation_rle"])\n\n    represented_stems = len(stem_to_rows)\n    print(f"  Represented Disks: {represented_stems} (out of {expected_test_stems})")\n    assert represented_stems <= expected_test_stems, \\\n        f"FATAL: Represented stems ({represented_stems}) exceeds expected test stems ({expected_test_stems})!"\n\n    # 4. Check inference manifest and/or test directory if available\n    manifest_p = csv_path.parent / "inference_manifest.json"\n    if manifest_p.exists():\n        import json\n        with open(manifest_p, "r", encoding="utf-8") as mf:\n            man = json.load(mf)\n        assert man.get("total_test_images") == expected_test_stems, \\\n            f"FATAL: Manifest total_test_images ({man.get(\'total_test_images\')}) != {expected_test_stems}!"\n        assert man.get("represented_disks") == represented_stems, \\\n            f"FATAL: Manifest represented_disks ({man.get(\'represented_disks\')}) != {represented_stems}!"\n        print(f"  [MANIFEST VERIFIED] Verified against inference_manifest.json ({expected_test_stems} test images).")\n\n    # If test directory is discoverable, assert all represented stems are legitimate test stems\n    candidate_test_dirs = [\n        csv_path.parent / "test_images",\n        Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images"),\n        Path("/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026/test/test_images"),\n        Path("data/MAGFiLO_1.0_Kaggle_2026/test/test_images"),\n    ]\n    discovered_test_dir = next((td for td in candidate_test_dirs if td.exists()), None)\n    if discovered_test_dir and expected_test_stems == 180:\n        found_test_files = list(discovered_test_dir.glob("*.jpeg")) + list(discovered_test_dir.glob("*.jpg"))\n        assert len(found_test_files) == 180, f"FATAL: Discovered test dir has {len(found_test_files)} images, expected 180!"\n        test_stem_set = {p.stem for p in found_test_files}\n        unknown_stems = set(stem_to_rows.keys()) - test_stem_set\n        assert len(unknown_stems) == 0, f"FATAL: Found {len(unknown_stems)} unknown stems not in test set: {list(unknown_stems)[:3]}"\n        print(f"  [TEST SET VERIFIED] All {represented_stems} represented stems belong to official 180 test images.")\n    zero_area_count = 0\n    invalid_rle_count = 0\n    wrong_shape_count = 0\n    pairwise_overlap_count = 0\n    mask_areas = []\n\n    print("  Decoding masks & verifying zero-overlap contract...")\n    for stem, rles in stem_to_rows.items():\n        disk_masks = []\n        for rle_str in rles:\n            try:\n                m = decode_rle(rle_str, h=expected_shape[0], w=expected_shape[1])\n            except Exception as e:\n                invalid_rle_count += 1\n                continue\n\n            if m.shape != expected_shape:\n                wrong_shape_count += 1\n\n            area = int(m.sum())\n            if area <= 0:\n                zero_area_count += 1\n            else:\n                mask_areas.append(area)\n                disk_masks.append(m)\n\n        # Assert zero pairwise overlap on this disk\n        n_m = len(disk_masks)\n        for i in range(n_m):\n            for j in range(i + 1, n_m):\n                shared = int(np.logical_and(disk_masks[i], disk_masks[j]).sum())\n                if shared > 0:\n                    pairwise_overlap_count += 1\n\n    # Print distribution stats\n    rows_per_disk = [len(rles) for rles in stem_to_rows.values()]\n\n    report = {\n        "csv_path": str(csv_path),\n        "sha256": sha256_hash,\n        "total_rows": len(df),\n        "represented_disks": represented_stems,\n        "expected_test_stems": expected_test_stems,\n        "duplicate_filament_ids": n_dups,\n        "zero_area_count": zero_area_count,\n        "invalid_rle_count": invalid_rle_count,\n        "wrong_shape_count": wrong_shape_count,\n        "pairwise_overlap_violations": pairwise_overlap_count,\n        "rows_per_disk_stats": {\n            "min": int(np.min(rows_per_disk)) if rows_per_disk else 0,\n            "mean": float(np.mean(rows_per_disk)) if rows_per_disk else 0.0,\n            "median": float(np.median(rows_per_disk)) if rows_per_disk else 0.0,\n            "max": int(np.max(rows_per_disk)) if rows_per_disk else 0,\n        },\n        "mask_area_stats": {\n            "min": int(np.min(mask_areas)) if mask_areas else 0,\n            "q25": float(np.percentile(mask_areas, 25)) if mask_areas else 0.0,\n            "median": float(np.median(mask_areas)) if mask_areas else 0.0,\n            "q75": float(np.percentile(mask_areas, 75)) if mask_areas else 0.0,\n            "max": int(np.max(mask_areas)) if mask_areas else 0,\n        },\n        "contract_passed": (\n            n_dups == 0\n            and zero_area_count == 0\n            and invalid_rle_count == 0\n            and wrong_shape_count == 0\n            and pairwise_overlap_count == 0\n        ),\n    }\n\n    print("\\n" + "=" * 75)\n    print("AUDIT RESULTS SUMMARY:")\n    print(f"  Zero-area masks:               {zero_area_count} (PASS if 0)")\n    print(f"  Invalid RLE strings:           {invalid_rle_count} (PASS if 0)")\n    print(f"  Wrong-shape masks:             {wrong_shape_count} (PASS if 0)")\n    print(f"  Pairwise overlap violations:   {pairwise_overlap_count} (PASS if 0)")\n    print(f"  Rows/disk distribution:        mean={report[\'rows_per_disk_stats\'][\'mean\']:.1f}, median={report[\'rows_per_disk_stats\'][\'median\']:.1f}, max={report[\'rows_per_disk_stats\'][\'max\']}")\n    print(f"  Mask area distribution (px):   median={report[\'mask_area_stats\'][\'median\']:.0f}, min={report[\'mask_area_stats\'][\'min\']}, max={report[\'mask_area_stats\'][\'max\']}")\n    print(f"  CONTRACT PASSED:               {report[\'contract_passed\']}")\n    print("=" * 75)\n\n    assert report["contract_passed"], f"FATAL: Submission failed contract checks! {report}"\n    return report\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Audit Submission CSV Contract")\n    parser.add_argument("csv_path", type=str)\n    args = parser.parse_args()\n\n    audit_submission_csv(Path(args.csv_path))\n')

print("✅ Successfully deployed metrics/ and moonshot_2048/ modules to working directory.")


In [ ]:
# ==============================================================================
# CELL 5: Stage 0 — Dataset Conversion & Grouped Split Verification
# ==============================================================================
from pathlib import Path
import subprocess, sys

out_yolo = Path("/kaggle/working/data/yolo_native2048")
cmd = [
    sys.executable, "-m", "moonshot_2048.data",
    "--data-dir", str(data_dir),
    "--out-dir", str(out_yolo),
    "--val-fold", "0",
    "--n-splits", "5",
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)

yaml_path = out_yolo / "data.yaml"
assert yaml_path.exists(), f"FATAL: data.yaml not created at {yaml_path}"
with open(yaml_path) as f:
    print(f.read())


In [ ]:
# ==============================================================================
# CELL 6: Stage 1 — Native-2048 YOLOv8l-seg Training (Fallback Sequence)
# ==============================================================================
import time
from pathlib import Path
from moonshot_2048.train_yolov8l import train_yolov8l_with_fallback
from moonshot_2048.config import MoonshotConfig

t0_train = time.perf_counter()
yaml_p = Path("/kaggle/working/data/yolo_native2048/data.yaml")

best_ckpt, exp_manifest = train_yolov8l_with_fallback(
    data_yaml=yaml_p,
    epochs=MoonshotConfig.EPOCHS,
    patience=MoonshotConfig.PATIENCE,
    device=MoonshotConfig.DEVICE,
    dry_run=False,
)
print(f"\nTraining completed in {(time.perf_counter() - t0_train) / 60:.1f} minutes.")
print(f"Verified Best Checkpoint: {best_ckpt} (exists: {Path(best_ckpt).exists()})")


In [ ]:
# ==============================================================================
# CELL 7: Stage 2 — Multi-Annotator Kirillov PQ Sweep on Physical Disks
# ==============================================================================
from pathlib import Path
from ultralytics import YOLO
from moonshot_2048.train_yolov8l import run_holdout_validation_sweep
from moonshot_2048.config import MoonshotConfig

# Directive 18: Pass the direct Stage 1 checkpoint without arbitrary input discovery
assert best_ckpt is not None and Path(best_ckpt).exists(), f"FATAL: Checkpoint {best_ckpt} not found!"
print(f"Evaluating Direct Stage 1 Checkpoint: {best_ckpt}")

model = YOLO(str(best_ckpt))
sweep_results = run_holdout_validation_sweep(
    model=model,
    magfilo_dir=data_dir,
    yolo_data_dir=Path("/kaggle/working/data/yolo_native2048"),
    nms_iou=MoonshotConfig.NMS_IOU,  # Deployment parity (iou=0.00)
    imgsz=MoonshotConfig.IMGSZ,
    device="0",
)


In [ ]:
# ==============================================================================
# CELL 8: Checkpoint Artifact Inspection & Experiment Manifest
# ==============================================================================
import json, hashlib
from pathlib import Path

assert Path(best_ckpt).exists(), f"FATAL: Checkpoint not found at {best_ckpt}"
sha256 = hashlib.sha256(Path(best_ckpt).read_bytes()).hexdigest()
size_mb = Path(best_ckpt).stat().st_size / (1024 ** 2)

print("=" * 75)
print("TRAINED CHECKPOINT ARTIFACT VERIFICATION & EXPERIMENT MANIFEST")
print("=" * 75)
print(f"  Checkpoint Path: {best_ckpt}")
print(f"  File Size:       {size_mb:.2f} MB")
print(f"  SHA256 Hash:     {sha256}")
print("  Experiment Manifest Content:")
print(json.dumps(exp_manifest, indent=2))
print("=" * 75)
